# 04 三次样条与 B 样条数值微分

直接差分只使用局部数据。样条微分的思路不同：先构造一个分段多项式函数，再对这个函数逐段解析求导。这样可以自然处理非等距节点，并获得更平滑的导数曲线。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    cubic_uniform_b_spline_basis,
    cubic_uniform_b_spline_basis_derivative,
    differentiate_discrete,
    natural_cubic_spline_derivative,
    uniform_b_spline_curve_derivative,
)


## 自然三次样条的分段形式

在区间 $[x_i,x_{i+1}]$ 上，自然三次样条写作

$$
S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3.
$$

因此

$$
S_i'(x)=b_i+2c_i(x-x_i)+3d_i(x-x_i)^2,
$$

$$
S_i''(x)=2c_i+6d_i(x-x_i).
$$

本节重点是从样条系数到导数，而不是只调用现成库。


In [ ]:
def teaching_spline_derivative_from_coefficients(x, a, b, c, d, x_eval, order=1):
    x_eval = np.asarray(x_eval, dtype=float)
    indices = np.searchsorted(x, x_eval, side="right") - 1
    indices = np.clip(indices, 0, len(a) - 1)
    dx = x_eval - x[indices]
    if order == 1:
        return b[indices] + 2 * c[indices] * dx + 3 * d[indices] * dx**2
    if order == 2:
        return 2 * c[indices] + 6 * d[indices] * dx
    raise ValueError("order must be 1 or 2")


## 教学实验：样条导数和直接差分

下面用非等距节点采样 $\sin x$。直接差分只在节点上估计导数，样条微分可以在区间内部连续计算导数。


In [ ]:
x = np.array([0.0, 0.25, 0.8, 1.4, 2.2, 3.1, 4.0, 5.0, 2 * math.pi])
y = np.sin(x)
x_eval = np.linspace(0.0, 2 * math.pi, 300)

spline_d1 = natural_cubic_spline_derivative(x, y, x_eval, derivative_order=1)
spline_d2 = natural_cubic_spline_derivative(x, y, x_eval, derivative_order=2)
node_diff = differentiate_discrete(x, y, stencil_size=5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_eval, np.cos(x_eval), label="解析一阶导数")
axes[0].plot(x_eval, spline_d1, "--", label="样条一阶导数")
axes[0].scatter(x, node_diff, s=25, label="节点差分")
axes[0].set_title("一阶导数")
axes[1].plot(x_eval, -np.sin(x_eval), label="解析二阶导数")
axes[1].plot(x_eval, spline_d2, "--", label="样条二阶导数")
axes[1].set_title("二阶导数")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.show()


## 与 SciPy CubicSpline 对照

`py_sc` 中的实现使用第二章的自然三次样条。若环境中安装了 SciPy，可以与 `scipy.interpolate.CubicSpline(..., bc_type="natural")` 的导数结果对照。这里把 SciPy 作为验证工具，而不是替代本章的推导。


In [ ]:
try:
    from scipy.interpolate import CubicSpline
except ImportError:
    CubicSpline = None

if CubicSpline is None:
    print("当前环境未安装 SciPy，跳过 CubicSpline 对照。")
else:
    scipy_spline = CubicSpline(x, y, bc_type="natural")
    scipy_d1 = scipy_spline(x_eval, 1)
    scipy_d2 = scipy_spline(x_eval, 2)
    print("一阶导数最大差异：", np.max(np.abs(spline_d1 - scipy_d1)))
    print("二阶导数最大差异：", np.max(np.abs(spline_d2 - scipy_d2)))


## 边界条件的影响

自然三次样条要求两端二阶导数为零。如果真实函数在端点的二阶导数并不接近零，端点附近的导数误差可能较大。固定导数边界、not-a-knot 边界和平滑样条都可能改变导数结果。本轮公共实现先保持自然边界，与第二章已有代码一致。


In [ ]:
for n in [9, 17, 33, 65]:
    xs = np.linspace(0.0, math.pi, n)
    ys = np.sin(xs)
    d1 = natural_cubic_spline_derivative(xs, ys, xs)
    interior = slice(1, -1)
    err = np.max(np.abs(d1[interior] - np.cos(xs[interior])))
    print(f"n={n:2d}  内部节点一阶导数最大误差={err:.3e}")


## 三次均匀 B 样条导数

三次均匀 B 样条基函数可写为

$$
B(x)=\frac{1}{6}\sum_{j=0}^4(-1)^j\binom{4}{j}(x-j)_+^3.
$$

因此 $B'(x)$ 和 $B''(x)$ 可由截断幂直接求导得到。B 样条的局部支撑性意味着一个位置的导数只依赖少数控制系数。


In [ ]:
t = np.linspace(-0.5, 4.5, 400)
B = cubic_uniform_b_spline_basis(t)
B1 = cubic_uniform_b_spline_basis_derivative(t, derivative_order=1)
B2 = cubic_uniform_b_spline_basis_derivative(t, derivative_order=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, B, label="B(x)")
ax.plot(t, B1, label="B'(x)")
ax.plot(t, B2, label="B''(x)")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("三次均匀 B 样条基函数及其导数")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## B 样条曲线导数示例与待完善内容

若

$$
s(t)=\sum_j c_j B(t-j),
$$

则

$$
s'(t)=\sum_j c_j B'(t-j),\qquad s''(t)=\sum_j c_j B''(t-j).
$$

下面只展示给定控制系数时的导数计算。完整的 B 样条拟合、边界控制点求解和自适应节点策略仍属于后续待完善内容。


In [ ]:
control = np.array([0.0, 0.4, 1.0, 0.2, -0.3, 0.1, 0.0])
t_eval = np.linspace(0.0, len(control) + 3.0, 500)
curve_d1 = uniform_b_spline_curve_derivative(control, t_eval, derivative_order=1)
curve_d2 = uniform_b_spline_curve_derivative(control, t_eval, derivative_order=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_eval, curve_d1, label="曲线一阶导数")
ax.plot(t_eval, curve_d2, label="曲线二阶导数")
ax.set_title("给定控制系数的 B 样条导数示例")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 小结

样条微分的优势是把离散数据重构成分段多项式，再逐段解析求导。它适合非等距、相对光滑的数据。局限是插值样条会穿过所有数据点，因此含噪声数据上仍可能产生导数振荡；更工程化的处理通常需要平滑样条或正则化模型。
